In [ ]:
# In[1]:

import pandas as pd

# Read CSV files
metric_df = pd.read_csv('metric.csv')
log_df = pd.read_csv('log.csv')
trace_df = pd.read_csv('trace.csv')
error_df = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, log_df, trace_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build compact summary for a dataframe
def compact_summary(df, filename, kpi_col=None, check_message=False, list_limit=50):
    n_rows = len(df)
    min_ts = df['timestamp'].min() if 'timestamp' in df.columns else pd.NaT
    max_ts = df['timestamp'].max() if 'timestamp' in df.columns else pd.NaT
    # unique cmdb ids (limit)
    unique_cmdb = list(pd.unique(df['cmdb_id'])) if 'cmdb_id' in df.columns else []
    unique_cmdb_trunc = unique_cmdb[:list_limit]
    unique_cmdb_str = ', '.join(map(str, unique_cmdb_trunc))
    # unique KPI/field names
    if kpi_col and kpi_col in df.columns:
        unique_kpis = list(pd.unique(df[kpi_col]))
        unique_kpis_trunc = unique_kpis[:list_limit]
        unique_kpis_str = ', '.join(map(str, unique_kpis_trunc))
    else:
        unique_kpis_str = None
    # error_logs: whether 'message' exists
    has_message = None
    if check_message:
        has_message = ('message' in df.columns)
    # Format timestamps as ISO UTC strings (or None)
    min_ts_iso = min_ts.isoformat() if not pd.isnull(min_ts) else None
    max_ts_iso = max_ts.isoformat() if not pd.isnull(max_ts) else None
    summary = pd.DataFrame([{
        'file': filename,
        'n_rows': n_rows,
        'min_timestamp_utc': min_ts_iso,
        'max_timestamp_utc': max_ts_iso,
        'unique_cmdb_ids_sample': unique_cmdb_str,
        'unique_field_names_sample': unique_kpis_str,
        'message_column_exists': has_message
    }])
    head = df.head(5)
    return summary, head

metric_summary, metric_head = compact_summary(metric_df, 'metric.csv', kpi_col='kpi_name')
log_summary, log_head = compact_summary(log_df, 'log.csv', kpi_col='log_name')
trace_summary, trace_head = compact_summary(trace_df, 'trace.csv', kpi_col='trace_name')
error_summary, error_head = compact_summary(error_df, 'error_logs.csv', check_message=True)

# Display compact summaries and first 5 rows for each file (as variables)
metric_summary, metric_head, log_summary, log_head, trace_summary, trace_head, error_summary, error_head

```
Out[1]:
```
summary = (
    "metric.csv: 1897 rows; timestamps from 2024-01-21T13:47:00+00:00 to 2024-01-21T14:11:00+00:00. "
    "Contains multiple cmdb_id values (examples: carts, carts-db, catalogue, catalogue-db, front-end, …) and KPI names (examples: cpu, latency-50, latency-90, mem, socket, …). "
    "First 5 rows include per-minute metrics for the 'carts' service (cpu, latency-50, latency-90, mem, socket). "
    "\n\nlog.csv: 458 rows; timestamps from 2024-01-21T13:47:00+00:00 to 2024-01-21T14:11:00+00:00. "
    "cmdb_id examples: carts, catalogue, front-end, orders, payment, …. Log fields include log.error_count and log.row_count. "
    "First 5 rows show log.error_count=0 and log.row_count values for several services. "
    "\n\ntrace.csv: 0 rows (empty). No timestamps, cmdb_id values, or trace_name entries present. "
    "\n\nerror_logs.csv: 0 rows (empty) but the file includes a 'message' column (message column exists). No error log records were captured in the time range."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  n_rows          min_timestamp_utc          max_timestamp_utc                             unique_cmdb_ids_sample                          unique_field_names_sample message_column_exists
0  metric.csv    1897  2024-01-21T13:47:00+00:00  2024-01-21T14:11:00+00:00  carts, carts-db, catalogue, catalogue-db, fron...  cpu, latency-50, latency-90, mem, socket, work...                  None,                   timestamp cmdb_id    kpi_name         value
0 2024-01-21 13:47:00+00:00   carts         cpu  2.016814e+01
1 2024-01-21 13:47:00+00:00   carts  latency-50  1.727911e-02
2 2024-01-21 13:47:00+00:00   carts  latency-90  3.423244e-02
3 2024-01-21 13:47:00+00:00   carts         mem  2.086359e+08
4 2024-01-21 13:47:00+00:00   carts      socket  1.000000e+01,       file  n_rows          min_timestamp_utc          max_timestamp_utc                             unique_cmdb_ids_sample       unique_field_names_sample message_column_exists
0  log.csv     458  2024-01-21T13:47:00+00:00  2024-01-21T14:11:00+00:00  carts, catalogue, front-end, orders, payment, ...  log.error_count, log.row_count                  None,                   timestamp    cmdb_id         log_name  value
0 2024-01-21 13:47:00+00:00      carts  log.error_count      0
1 2024-01-21 13:47:00+00:00      carts    log.row_count     50
2 2024-01-21 13:47:00+00:00  catalogue  log.error_count      0
3 2024-01-21 13:47:00+00:00  catalogue    log.row_count     53
4 2024-01-21 13:47:00+00:00  front-end  log.error_count      0,         file  n_rows min_timestamp_utc max_timestamp_utc unique_cmdb_ids_sample unique_field_names_sample message_column_exists
0  trace.csv       0              None              None                                                                   None, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [],              file  n_rows min_timestamp_utc max_timestamp_utc unique_cmdb_ids_sample unique_field_names_sample  message_column_exists
0  error_logs.csv       0              None              None                                             None                   True, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Load metric.csv and ensure timestamp parsed as UTC datetimes
metric_df = pd.read_csv('metric.csv')
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Global stats (computed on the entire series BEFORE filtering)
global_stats = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(
        count_all='count',
        p95_value=lambda s: s.quantile(0.95),
        mean='mean',
        max='max'
    )
)

# Define incident window (UTC)
start_ts = pd.to_datetime("2024-01-21T13:44:38+00:00", utc=True)
end_ts   = pd.to_datetime("2024-01-21T14:14:38+00:00", utc=True)

# Filter metric rows to the incident window
metric_win = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Incident-level stats per (cmdb_id, kpi_name)
incident_stats = (
    metric_win
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        incident_count=('value', 'count'),
        earliest_incident_ts=('timestamp', 'min')
    )
)

# Prepare breach detection by joining p95 thresholds to windowed rows
metric_win_with_p95 = metric_win.merge(
    global_stats[['cmdb_id', 'kpi_name', 'p95_value']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Rows in window where value >= p95_value (breaches)
breaches = metric_win_with_p95[metric_win_with_p95['value'] >= metric_win_with_p95['p95_value']].copy()

breach_stats = (
    breaches
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        breach_count=('value', 'count'),
        earliest_breach_ts=('timestamp', 'min')
    )
)

# Merge all stats together
summary = (
    global_stats
    .merge(incident_stats, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(breach_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill missing numeric counts with 0 and keep timestamps as NaT where missing
summary['incident_count'] = summary['incident_count'].fillna(0).astype(int)
summary['breach_count'] = summary['breach_count'].fillna(0).astype(int)

# Format earliest timestamps as ISO strings, keep NaT as None
def to_iso_or_none(x):
    return x.isoformat() if pd.notnull(x) else None

summary['earliest_incident_ts'] = summary['earliest_incident_ts'].apply(to_iso_or_none)
summary['earliest_breach_ts'] = summary['earliest_breach_ts'].apply(to_iso_or_none)

# Select and order the requested columns
result_df = summary[
    [
        'cmdb_id', 'kpi_name',
        'count_all', 'p95_value', 'mean', 'max',
        'incident_count', 'earliest_incident_ts',
        'breach_count', 'earliest_breach_ts'
    ]
]

# Sort by breach_count desc then incident_count desc, limit to top 20 rows
result_df_top = result_df.sort_values(['breach_count', 'incident_count'], ascending=[False, False]).head(20)

# Totals
total_unique_cmdb = metric_df['cmdb_id'].nunique()
total_unique_kpi = metric_df['kpi_name'].nunique()

# Display compact results: top table and the two totals
result_df_top, total_unique_cmdb, total_unique_kpi

```
Out[2]:
```
Totals:
- Unique cmdb_id values: 15
- Unique kpi_name values: 8

High-level findings (sorted by breach frequency):
- Many metrics show every point in the incident window exceeded their global 95th-percentile (breach_count = incident_count = 25). These include:
  - catalogue — socket (incident start 2024-01-21T13:47:00+00:00)
  - catalogue-db — diskio (2024-01-21T13:47:00+00:00)
  - catalogue-db — socket (2024-01-21T13:47:00+00:00)
  - front-end — error (2024-01-21T13:47:00+00:00)
  - orders-db — socket (2024-01-21T13:47:00+00:00)
  - queue-master — socket (2024-01-21T13:47:00+00:00)
  - rabbitmq — socket (2024-01-21T13:47:00+00:00)
  - rabbitmq-exporter — mem (2024-01-21T13:47:00+00:00)
  - rabbitmq-exporter — socket (2024-01-21T13:47:00+00:00)
  - session-db — socket (2024-01-21T13:47:00+00:00)
  - user — error (2024-01-21T13:47:00+00:00)

- Several other components show substantial but partial breaches:
  - payment — socket: count_all=25, p95=5.0, incident_count=25, breach_count=12 (first breach 2024-01-21T14:00:00+00:00)
  - user-db — socket: count_all=25, p95=8.0, incident_count=25, breach_count=9 (first breach 2024-01-21T14:03:00+00:00)
  - carts-db — socket: count_all=25, p95=10.0, incident_count=25, breach_count=7 (first breach 2024-01-21T14:05:00+00:00)
  - user — socket: count_all=25, p95=17.0, incident_count=25, breach_count=3 (first breach 2024-01-21T14:09:00+00:00)
  - carts — cpu: count_all=25, p95≈77.74, mean≈38.62, max≈77.76, incident_count=25, breach_count=2 (first breach 2024-01-21T14:09:00+00:00)
  - carts — latency-50 and latency-90 and mem also show small numbers of breaches (each breach_count=2, earliest breaches between 14:04–14:05 UTC)

Interpretation:
- The most frequent anomalies are socket-related metrics and some disk I/O / error metrics across multiple services (catalogue, catalogue-db, rabbitmq, orders-db, queue-master, session-db, etc.). Many of these had every measurement in the incident window above their global p95 thresholds, indicating sustained elevated values during the window.
- Payment and user-related socket metrics also show substantial breaches but less uniformly.
- Carts shows elevated CPU and latency breaches but only a few points exceed the global p95.

If you want, I can return the full top-20 aggregated table rows shown earlier (with exact numeric columns) or filter this summary to only service-level conclusions.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count_all     p95_value          mean           max  incident_count       earliest_incident_ts  breach_count         earliest_breach_ts
15          catalogue      socket         25  6.000000e+00  6.000000e+00  6.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
18       catalogue-db      diskio         25  0.000000e+00  0.000000e+00  0.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
20       catalogue-db      socket         25  4.000000e+00  4.000000e+00  4.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
22          front-end       error         25  0.000000e+00  2.436364e-04  6.090909e-03              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
37          orders-db      socket         25  8.000000e+00  8.000000e+00  8.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
47       queue-master      socket         25  3.000000e+00  3.000000e+00  3.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
51           rabbitmq      socket         25  1.100000e+01  1.100000e+01  1.100000e+01              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
53  rabbitmq-exporter         mem         25  4.816896e+06  4.816896e+06  4.816896e+06              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
54  rabbitmq-exporter      socket         25  1.000000e+00  1.000000e+00  1.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
58         session-db      socket         25  3.000000e+00  3.000000e+00  3.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
66               user       error         25  0.000000e+00  0.000000e+00  0.000000e+00              25  2024-01-21T13:47:00+00:00            25  2024-01-21T13:47:00+00:00
1               carts      diskio         24  0.000000e+00  0.000000e+00  0.000000e+00              24  2024-01-21T13:48:00+00:00            24  2024-01-21T13:48:00+00:00
42            payment      socket         25  5.000000e+00  4.101333e+00  5.000000e+00              25  2024-01-21T13:47:00+00:00            12  2024-01-21T14:00:00+00:00
75            user-db      socket         25  8.000000e+00  7.351576e+00  8.000000e+00              25  2024-01-21T13:47:00+00:00             9  2024-01-21T14:03:00+00:00
10           carts-db      socket         25  1.000000e+01  7.310000e+00  1.000000e+01              25  2024-01-21T13:47:00+00:00             7  2024-01-21T14:05:00+00:00
70               user      socket         25  1.700000e+01  1.428055e+01  1.700000e+01              25  2024-01-21T13:47:00+00:00             3  2024-01-21T14:09:00+00:00
0               carts         cpu         25  7.739564e+01  3.861917e+01  7.761950e+01              25  2024-01-21T13:47:00+00:00             2  2024-01-21T14:09:00+00:00
2               carts  latency-50         25  1.992910e-02  1.395071e-02  2.208539e-02              25  2024-01-21T13:47:00+00:00             2  2024-01-21T14:05:00+00:00
3               carts  latency-90         25  1.579987e-01  6.772411e-02  2.073788e-01              25  2024-01-21T13:47:00+00:00             2  2024-01-21T14:04:00+00:00
4               carts         mem         25  1.893660e+09  8.269865e+08  1.969124e+09              25  2024-01-21T13:47:00+00:00             2  2024-01-21T14:05:00+00:00, 15, 8)```
```

In [ ]:
# In[3]:

import pandas as pd

# Load or reuse log_df and ensure timestamp parsed as UTC datetimes
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure timestamp is datetime64[ns, UTC]
if not pd.api.types.is_datetime64tz_dtype(log_df['timestamp']):
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-21T13:44:38+00:00", utc=True)
end_ts   = pd.to_datetime("2024-01-21T14:14:38+00:00", utc=True)

# Filter logs to the incident window
log_win = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# Total number of log rows in-window
total_in_window = int(log_win.shape[0])

# 1) Aggregate table grouped by (cmdb_id, log_name)
agg = (
    log_win
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        incident_count=('value', 'count'),
        sum_value=('value', 'sum'),
        max_value=('value', 'max'),
        earliest_ts_in_window=('timestamp', 'min')
    )
)

# Format earliest timestamp as ISO UTC string, keep None for NaT
def to_iso_or_none(x):
    return x.isoformat() if pd.notnull(x) else None

agg['earliest_ts_in_window'] = agg['earliest_ts_in_window'].apply(to_iso_or_none)

agg_sorted = agg.sort_values(['sum_value', 'incident_count'], ascending=[False, False]).head(20)

# 2) Sample up to first 20 raw rows where log_name == 'log.error_count' and value > 0
error_samples = (
    log_win[(log_win['log_name'] == 'log.error_count') & (log_win['value'] > 0)]
    .sort_values('timestamp')
    .head(20)
    .copy()
)

# Format timestamp in samples as ISO strings for compact display
if not error_samples.empty:
    error_samples['timestamp'] = error_samples['timestamp'].apply(lambda x: x.isoformat())

# Return the two outputs and the total count
agg_sorted, error_samples[['timestamp', 'cmdb_id', 'log_name', 'value']], total_in_window

```
Out[3]:
```
Summary of log.csv within the incident window [2024-01-21T13:44:38+00:00, 2024-01-21T14:14:38+00:00]:

Total log rows in-window: 458

1) Top aggregated (cmdb_id, log_name) by sum_value (top rows shown)
- front-end — log.row_count: incident_count=25, sum_value=47562, max_value=2101, earliest_ts=2024-01-21T13:47:00+00:00
- user — log.row_count: incident_count=25, sum_value=14324, max_value=634, earliest_ts=2024-01-21T13:47:00+00:00
- queue-master — log.row_count: incident_count=25, sum_value=13410, max_value=612, earliest_ts=2024-01-21T13:47:00+00:00
- catalogue — log.row_count: incident_count=25, sum_value=3909, max_value=171, earliest_ts=2024-01-21T13:47:00+00:00
- orders — log.row_count: incident_count=25, sum_value=2980, max_value=136, earliest_ts=2024-01-21T13:47:00+00:00
- payment — log.row_count: incident_count=25, sum_value=2425, max_value=108, earliest_ts=2024-01-21T13:47:00+00:00
- carts — log.row_count: incident_count=23, sum_value=1555, max_value=200, earliest_ts=2024-01-21T13:47:00+00:00
- shipping — log.row_count: incident_count=25, sum_value=1490, max_value=68, earliest_ts=2024-01-21T13:47:00+00:00
- carts-db — log.row_count: incident_count=24, sum_value=29, max_value=3, earliest_ts=2024-01-21T13:48:00+00:00
- session-db — log.row_count: incident_count=5, sum_value=25, max_value=5, earliest_ts=2024-01-21T13:49:00+00:00
- user-db — log.row_count: incident_count=2, sum_value=3, max_value=2, earliest_ts=2024-01-21T13:47:00+00:00

Also present in the top table: log.error_count entries for many services — all have incident_count matching window length (23–25) but sum_value = 0 and max_value = 0 (i.e., no errors recorded).

2) Sample of raw log.error_count > 0 rows:
- No rows match (there are zero log.error_count entries with value > 0 in the filtered window).

Interpretation:
- The dominant log activity in the window is log.row_count (high traffic/row counts), especially for front-end and user services.
- No recorded log.error_count > 0 during the incident window across services (all error_count sums and maxes are zero).

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  incident_count  sum_value  max_value      earliest_ts_in_window
7      front-end    log.row_count              25      47562       2101  2024-01-21T13:47:00+00:00
19          user    log.row_count              25      14324        634  2024-01-21T13:47:00+00:00
13  queue-master    log.row_count              25      13410        612  2024-01-21T13:47:00+00:00
5      catalogue    log.row_count              25       3909        171  2024-01-21T13:47:00+00:00
9         orders    log.row_count              25       2980        136  2024-01-21T13:47:00+00:00
11       payment    log.row_count              25       2425        108  2024-01-21T13:47:00+00:00
1          carts    log.row_count              23       1555        200  2024-01-21T13:47:00+00:00
17      shipping    log.row_count              25       1490         68  2024-01-21T13:47:00+00:00
3       carts-db    log.row_count              24         29          3  2024-01-21T13:48:00+00:00
15    session-db    log.row_count               5         25          5  2024-01-21T13:49:00+00:00
21       user-db    log.row_count               2          3          2  2024-01-21T13:47:00+00:00
4      catalogue  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
6      front-end  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
8         orders  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
10       payment  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
12  queue-master  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
16      shipping  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
18          user  log.error_count              25          0          0  2024-01-21T13:47:00+00:00
2       carts-db  log.error_count              24          0          0  2024-01-21T13:48:00+00:00
0          carts  log.error_count              23          0          0  2024-01-21T13:47:00+00:00, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [], 458)```
```